# Experiment 2 — Left ICA stenosis sweep with pressure inlets

This experiment progressively reduces the reference area of `L-ICA_I` and compares regional perfusion against the healthy baseline.

Unlike prescribed velocity inlets, pressure inlets allow flow to respond to the stenosis. The notebook uses `hemo1d.create_arterial_pressure_inflow(...)` from the source package.

## References used for interpretation

- Normal adult cerebral blood flow is commonly summarized as about 750 mL/min, or about 50–54 mL/100 g/min. Ischemic concern is often discussed below about 18–20 mL/100 g/min, with severe injury at lower values.
- The Circle of Willis is a collateral network: under unilateral carotid stenosis, the anterior communicating artery can provide right-to-left or left-to-right compensation, while posterior communicating arteries may recruit posterior-anterior compensation depending on anatomy.
- In this project, the six 0D terminal beds are not meant to reproduce exact patient-specific tissue territories. They are a controlled physiological calibration that converts 1D outlet flows into regional perfusion metrics.


In [1]:

from __future__ import annotations

import copy
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import hemo1d as hd

BASE_CONFIG_PATH = Path("./configs/CoW_normal.json")
GENERATED_CONFIG_DIR = Path("./configs/generated_notebooks")
OUTPUT_ROOT = Path("./outputs/notebooks")

MMHG_TO_DYN_CM2 = 1333.22

# Solver settings. Increase T_END / decrease DT for final polished results.
METHOD = "CG"
DG_TIME_SCHEME = "rk2"
DEGREE = 1
H = 0.0625
DT = 1.0e-5
T_END = 2.0
RECORD_EVERY = 10
SHOW_PLOTS = False

HEART_RATE = 1.2    # Hz, about 72 bpm
RAMP_TIME = 0.20    # seconds

# Six tissue territories represented by six 0D capillary beds.
EXPECTED_BEDS = {
    "L-ACA_bed": {"target_flow_ml_min": 100.0, "target_perfusion": 50.0, "target_pcap_mmhg": 35.0},
    "R-ACA_bed": {"target_flow_ml_min": 100.0, "target_perfusion": 50.0, "target_pcap_mmhg": 35.0},
    "L-MCA_bed": {"target_flow_ml_min": 190.0, "target_perfusion": 50.0, "target_pcap_mmhg": 35.0},
    "R-MCA_bed": {"target_flow_ml_min": 190.0, "target_perfusion": 50.0, "target_pcap_mmhg": 35.0},
    "L-PCA_bed": {"target_flow_ml_min": 85.0,  "target_perfusion": 50.0, "target_pcap_mmhg": 35.0},
    "R-PCA_bed": {"target_flow_ml_min": 85.0,  "target_perfusion": 50.0, "target_pcap_mmhg": 35.0},
}

STANDARD_CAPILLARY_BEDS = {
    "L-ACA_bed": {
        "outlets": [{"vessel_id": "L-ACA_II", "side": "right", "R_art": 3.600e4}],
        "C": 6.95e-6, "R_ven": 2.160e4, "P_ven": 1.0666e4,
        "P0": 4.6663e4, "tissue_volume": 200.0,
    },
    "R-ACA_bed": {
        "outlets": [{"vessel_id": "R-ACA_II", "side": "right", "R_art": 3.600e4}],
        "C": 6.95e-6, "R_ven": 2.160e4, "P_ven": 1.0666e4,
        "P0": 4.6663e4, "tissue_volume": 200.0,
    },
    "L-MCA_bed": {
        "outlets": [{"vessel_id": "L-MCA", "side": "right", "R_art": 1.895e4}],
        "C": 1.32e-5, "R_ven": 1.137e4, "P_ven": 1.0666e4,
        "P0": 4.6663e4, "tissue_volume": 380.0,
    },
    "R-MCA_bed": {
        "outlets": [{"vessel_id": "R-MCA", "side": "right", "R_art": 1.895e4}],
        "C": 1.32e-5, "R_ven": 1.137e4, "P_ven": 1.0666e4,
        "P0": 4.6663e4, "tissue_volume": 380.0,
    },
    "L-PCA_bed": {
        "outlets": [{"vessel_id": "L-PCA_II", "side": "right", "R_art": 4.235e4}],
        "C": 5.90e-6, "R_ven": 2.541e4, "P_ven": 1.0666e4,
        "P0": 4.6663e4, "tissue_volume": 170.0,
    },
    "R-PCA_bed": {
        "outlets": [{"vessel_id": "R-PCA_II", "side": "right", "R_art": 4.235e4}],
        "C": 5.90e-6, "R_ven": 2.541e4, "P_ven": 1.0666e4,
        "P0": 4.6663e4, "tissue_volume": 170.0,
    },
}

PROBE_VESSELS = (
    "BAS", "L-ICA_I", "R-ICA_I", "ACA", "L-PCommA", "R-PCommA",
    "L-MCA", "R-MCA", "L-ACA_II", "R-ACA_II", "L-PCA_II", "R-PCA_II",
)


def load_base_config() -> dict:
    if not BASE_CONFIG_PATH.exists():
        raise FileNotFoundError(
            f"Could not find {BASE_CONFIG_PATH}. Run this notebook from the repository root."
        )
    with BASE_CONFIG_PATH.open() as f:
        return json.load(f)


def write_config(config: dict, name: str) -> Path:
    GENERATED_CONFIG_DIR.mkdir(parents=True, exist_ok=True)
    path = GENERATED_CONFIG_DIR / name
    with path.open("w") as f:
        json.dump(config, f, indent=2)
    return path


def prepare_base_cow_config() -> dict:
    """Return a copy of the CoW config with physiological p0 and six capillary beds."""
    config = copy.deepcopy(load_base_config())
    config.setdefault("defaults", {})
    config["defaults"]["p0"] = 80.0 * MMHG_TO_DYN_CM2
    config["defaults"]["p_ext"] = 0.0
    config["capillary_beds"] = copy.deepcopy(STANDARD_CAPILLARY_BEDS)
    return config


def set_pressure_inlets(model) -> None:
    """Use the source-code arterial pressure waveform factory, not a notebook-local one."""
    model.set_inlet(
        vessel_id="L-ICA_I",
        kind="pressure",
        function=hd.create_arterial_pressure_inflow(
            mean_mmhg=85.0,
            amp1_mmhg=14.0,
            amp2_mmhg=4.0,
            heart_rate=HEART_RATE,
            ramp_time=RAMP_TIME,
            phase_s=0.00,
        ),
    )
    model.set_inlet(
        vessel_id="R-ICA_I",
        kind="pressure",
        function=hd.create_arterial_pressure_inflow(
            mean_mmhg=85.0,
            amp1_mmhg=14.0,
            amp2_mmhg=4.0,
            heart_rate=HEART_RATE,
            ramp_time=RAMP_TIME,
            phase_s=0.00,
        ),
    )
    model.set_inlet(
        vessel_id="BAS",
        kind="pressure",
        function=hd.create_arterial_pressure_inflow(
            mean_mmhg=84.0,
            amp1_mmhg=11.0,
            amp2_mmhg=3.0,
            heart_rate=HEART_RATE,
            ramp_time=RAMP_TIME,
            phase_s=0.04,
        ),
    )


def configure_model(config_path: Path):
    model = hd.load_from_config(config_path)
    set_pressure_inlets(model)
    model.set_solver(
        method=METHOD,
        h=H,
        dt=DT,
        poly_order=DEGREE,
        dg_time_scheme=DG_TIME_SCHEME,
        record_every=RECORD_EVERY,
    )
    for vessel_id in PROBE_VESSELS:
        if vessel_id not in model.config.vessels:
            continue
        length = model.config.vessel(vessel_id).length
        model.add_probe(vessel_id=vessel_id, position=0.5 * length, name="mid")
    return model


def late_mask(times: np.ndarray, fraction: float = 0.25) -> np.ndarray:
    tail_start = times[-1] - fraction * (times[-1] - times[0])
    return times >= tail_start


def summarize_beds(results) -> pd.DataFrame:
    rows = []
    for bed_id, expected in EXPECTED_BEDS.items():
        samples = results.capillary_bed_history(bed_id)
        times = np.array([s.time for s in samples])
        tail = late_mask(times)
        pcap_mmhg = np.array([s.pressure for s in samples]) / MMHG_TO_DYN_CM2
        inflow_ml_min = np.array([s.total_inflow for s in samples]) * 60.0
        venous_ml_min = np.array([s.venous_outflow for s in samples]) * 60.0
        perfusion = np.array([s.regional_perfusion for s in samples]) * 6000.0
        rows.append({
            "bed": bed_id,
            "target_pcap_mmhg": expected["target_pcap_mmhg"],
            "pcap_mmhg": float(np.mean(pcap_mmhg[tail])),
            "pcap_min_tail": float(np.min(pcap_mmhg[tail])),
            "pcap_max_tail": float(np.max(pcap_mmhg[tail])),
            "target_flow_ml_min": expected["target_flow_ml_min"],
            "flow_ml_min": float(np.mean(inflow_ml_min[tail])),
            "venous_ml_min": float(np.mean(venous_ml_min[tail])),
            "target_perfusion": expected["target_perfusion"],
            "perfusion": float(np.mean(perfusion[tail])),
        })
    return pd.DataFrame(rows)


def probe_late_mean_flow(results, vessel_id: str, probe_name: str = "mid") -> float | None:
    samples = results.history.probes.by_vessel_and_name(vessel_id, probe_name)
    if not samples:
        return None
    times = np.array([s.time for s in samples])
    tail = late_mask(times)
    flow_ml_min = np.array([s.flow_rate for s in samples]) * 60.0
    return float(np.mean(flow_ml_min[tail]))


def summarize_probe_flows(results) -> pd.DataFrame:
    rows = []
    for vessel_id in PROBE_VESSELS:
        q = probe_late_mean_flow(results, vessel_id)
        if q is not None:
            rows.append({"vessel": vessel_id, "mean_flow_ml_min": q})
    return pd.DataFrame(rows)


def run_case(config: dict, config_name: str, output_subdir: str, *, show_progress: bool = True):
    config_path = write_config(config, config_name)
    model = configure_model(config_path)
    results = model.solve(t_end=T_END, show_progress=show_progress)
    output_dir = OUTPUT_ROOT / output_subdir
    results.save(output_dir)
    results.plot_capillary_beds(output_dir / "plots", show=SHOW_PLOTS)
    return results, output_dir


## Define the stenosis sweep

This is an **area-stenosis surrogate**, not a full localized pressure-loss model.

For a stenosis level `s`, the generated config uses:


```text
A_new = (1 - s) A_original
```

This narrows the whole `L-ICA_I` segment. It is sufficient for comparing collateral trends, but a future localized stenosis/loss model would instead add a short nonlinear pressure-drop element.


In [2]:
L_ICA_BASE_AREA = 0.1252

STENOSIS_LEVELS = {
    "normal": 0.00,
    "LICA_area30": 0.30,
    "LICA_area50": 0.50,
    "LICA_area70": 0.70,
    "LICA_area85": 0.85,
}


def make_lica_stenosis_config(area_stenosis: float) -> dict:
    config = prepare_base_cow_config()
    config["vessels"]["L-ICA_I"]["initial_area"] = L_ICA_BASE_AREA * (1.0 - area_stenosis)
    return config

## Run all cases

Expected qualitative behavior:

- `L-MCA_bed` and `L-ACA_bed` should lose perfusion as left ICA stenosis increases.
- `R-MCA_bed` and `R-ACA_bed` should be comparatively preserved.
- Communicating-vessel flows should increase in magnitude when collateral compensation activates.


In [ ]:
case_tables = {}
probe_tables = {}

for case_name, stenosis in STENOSIS_LEVELS.items():
    print(f"\nRunning {case_name}: L-ICA_I area stenosis = {100*stenosis:.0f}%")
    config = make_lica_stenosis_config(stenosis)
    results, output_dir = run_case(
        config,
        config_name=f"02_CoW_{case_name}.json",
        output_subdir=f"02_LICA_stenosis/{case_name}",
    )
    bed_table = summarize_beds(results)
    bed_table["case"] = case_name
    bed_table["area_stenosis_pct"] = 100.0 * stenosis
    case_tables[case_name] = bed_table
    probe_table = summarize_probe_flows(results)
    probe_table["case"] = case_name
    probe_table["area_stenosis_pct"] = 100.0 * stenosis
    probe_tables[case_name] = probe_table
    print(f"Saved outputs to {output_dir}")

all_beds = pd.concat(case_tables.values(), ignore_index=True)
all_probes = pd.concat(probe_tables.values(), ignore_index=True)


Running normal: L-ICA_I area stenosis = 0%


Solving network: 0%|          | 0.00000/2.00000 [00:00<?]

## Regional perfusion table


In [ ]:
all_beds.pivot_table(
    index=["area_stenosis_pct", "case"],
    columns="bed",
    values="perfusion",
)

## Perfusion preservation ratio

A value of 1.0 means unchanged relative to the simulated healthy case.


In [ ]:
normal = all_beds[all_beds["case"] == "normal"].set_index("bed")["perfusion"]
ratio_rows = []
for _, row in all_beds.iterrows():
    ratio_rows.append({
        "case": row["case"],
        "area_stenosis_pct": row["area_stenosis_pct"],
        "bed": row["bed"],
        "perfusion_ratio": row["perfusion"] / normal.loc[row["bed"]],
    })
ratios = pd.DataFrame(ratio_rows)
ratios.pivot_table(index=["area_stenosis_pct", "case"], columns="bed", values="perfusion_ratio")

In [ ]:
plot_df = all_beds[all_beds["bed"].isin(["L-MCA_bed", "L-ACA_bed", "R-MCA_bed", "R-ACA_bed"])]
fig, ax = plt.subplots(figsize=(8, 4))
for bed_id, group in plot_df.groupby("bed"):
    group = group.sort_values("area_stenosis_pct")
    ax.plot(group["area_stenosis_pct"], group["perfusion"], marker="o", label=bed_id)
ax.axhline(50.0, linestyle="--", linewidth=1)
ax.axhspan(18.0, 20.0, alpha=0.15, label="ischemic concern band")
ax.set_xlabel("L-ICA area stenosis [%]")
ax.set_ylabel("Perfusion [mL/100g/min]")
ax.set_title("Experiment 2: regional perfusion vs L-ICA stenosis")
ax.legend()
plt.tight_layout()

## Communicating vessel / inlet flow diagnostics

These values help interpret whether the CoW is recruiting collateral pathways. Look especially at the change in `ACA`, `L-PCommA`, and `R-PCommA` magnitudes as stenosis increases.


In [ ]:
all_probes.pivot_table(
    index=["area_stenosis_pct", "case"],
    columns="vessel",
    values="mean_flow_ml_min",
)

## Discussion

A physiologically reasonable result is not necessarily that the left territories remain exactly at 50 mL/100g/min. The key comparison is the **trend**:

- moderate stenosis should be buffered by collateral flow,
- severe stenosis should reduce left anterior perfusion,
- communicating-artery flow should increase in magnitude,
- right-side territories should be less affected than left-side territories.

If the total flow does not change much with stenosis, the pressure source and terminal resistances may be dominating the solution too strongly; then increase stenosis severity or move toward a localized nonlinear loss model.
